# HDF5 Iceberg Metadata Provider

Verify the **standalone** `hdf5_iceberg` SDK:

1. **DatasetProvider** — read-only scan/audit of existing HDF5 (Layer A)
2. **MetadataProvider** — all writes under an isolated warehouse prefix (Layer B)
3. **register_root** — idempotent pointer-table + DCAT/SHACL TTL stubs

```python
from hdf5_iceberg import DatasetProvider, MetadataProvider, register_root
```

Copy this notebook to home if needed (server start usually seeds `/root/*.ipynb`).


In [ ]:
import sys
for _p in ("/root/sample-notebooks", "/app", "/root"):
    if _p not in sys.path:
        sys.path.insert(0, _p)
try:
    from cluster_env import load_cluster_config as _lcc
    _CFG = _lcc()
    print(_CFG.summary())
    import os as _os
    _os.environ.setdefault("S3_BUCKET", _CFG.s3_bucket)
    if _CFG.s3_endpoint:
        _os.environ.setdefault("S3_ENDPOINT", _CFG.s3_endpoint)
    _os.environ.setdefault("AWS_REGION", _CFG.s3_region)
except Exception as _e:
    print("cluster_env:", _e)

# --- Configuration ---
import os

# Layer A — existing HDF5 (RO). Lab default: product prefix under the data bucket.
DATA_ROOT = os.getenv(
    "HDF5_DATA_ROOT",
    f"s3://{os.getenv('S3_BUCKET', 'cyberphy')}/datasets/hdf5/cphy/",
)
# Layer B — isolated metadata warehouse (RW). Never under the data-only prefix if you can avoid it.
META_WAREHOUSE = os.getenv(
    "HDF5_META_WAREHOUSE",
    f"s3://{os.getenv('S3_BUCKET', 'cyberphy')}/cyberphy-md/iceberg/warehouse",
)
ADAPTER = os.getenv("HDF5_LAYOUT_ADAPTER", "product_prefix")  # or flat_prefix
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "http://127.0.0.1:9010")
MAX_FILES = int(os.getenv("HDF5_REGISTER_MAX", "24"))  # cap for lab demos
MIN_SIZE = int(os.getenv("HDF5_MIN_SIZE", "50000"))  # skip tiny leftovers

print(f"DATA_ROOT     = {DATA_ROOT}")
print(f"META_WAREHOUSE= {META_WAREHOUSE}")
print(f"ADAPTER       = {ADAPTER}")
print(f"S3_ENDPOINT   = {S3_ENDPOINT}")
print(f"MAX_FILES     = {MAX_FILES}")


In [ ]:
# --- Import standalone package (no cybersec/cyberphy prefix) ---
import sys
from pathlib import Path

# Monorepo / image / ConfigMap-adjacent layouts
for p in (
    Path("/app"),
    Path("/root"),
    Path.cwd(),
    Path("/app/packages/hdf5_iceberg/src"),
    Path.cwd() / "packages" / "hdf5_iceberg" / "src",
):
    if (p / "hdf5_iceberg").is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from hdf5_iceberg import DatasetProvider, MetadataProvider, register_root, __version__

print("hdf5_iceberg", __version__)
print("API:", DatasetProvider, MetadataProvider, register_root)


In [ ]:
# --- Providers ---
data = DatasetProvider(
    roots=[DATA_ROOT],
    readonly=True,
    adapter=ADAPTER,
    endpoint_url=S3_ENDPOINT or None,
    min_size_bytes=MIN_SIZE,
)
meta = MetadataProvider(
    warehouse=META_WAREHOUSE,
    endpoint_url=S3_ENDPOINT or None,
)

print("DatasetProvider roots:", list(data.roots), "adapter=", data.layout_adapter.name)
print("MetadataProvider warehouse:", meta.warehouse)
print("pointer table →", meta.pointer_table_uri())


In [ ]:
# --- Discover (list only, then optional audit sample) ---
cands = data.list_candidates(max_files=MAX_FILES)
print(f"candidates: {len(cands)}")
for c in cands[:5]:
    print(f"  {c.size_bytes/1e6:.2f} MB  tags={c.tags}  {c.uri}")

if cands:
    sample = data.audit(cands[0].uri)
    print("\\naudit sample:")
    print(f"  fingerprint={sample.fingerprint} uuid={sample.dataset_uuid}")
    print(f"  geometry={sample.n_series}×{sample.n_time} {sample.dtype} layout={sample.layout}")
    print(f"  t=[{sample.t_min_ns}, {sample.t_max_ns}]")
else:
    print("No candidates — generate lab HDF5 first or lower MIN_SIZE / change DATA_ROOT")


In [ ]:
# --- register_root (idempotent) ---
result = register_root(
    data,
    meta,
    max_files=MAX_FILES,
    emit_semantic=True,
)
print(result.summary())
if result.errors:
    print("ERRORS:", result.errors)
print("semantic:", result.semantic_uris)

# Second pass should reuse
result2 = register_root(data, meta, max_files=MAX_FILES, emit_semantic=True)
print("2nd pass:", result2.summary())
assert result2.n_registered == 0 or result2.n_reused >= result2.n_registered
print("idempotent OK")


In [ ]:
# --- Isolation check: warehouse keys only under META_WAREHOUSE ---
keys = meta.list_warehouse_keys(max_keys=40)
print(f"warehouse objects (first {len(keys)}):")
prefix = META_WAREHOUSE.replace("s3://", "").rstrip("/")
for k in keys:
    print(" ", k)
    # every key must contain META_WAREHOUSE path segment when using default warehouse
    assert prefix.split("/", 1)[-1][:12] in k or "hdf5_datasets" in k

# Refuse write outside warehouse
try:
    meta.assert_write_uri("s3://someone-elses-bucket/nope.parquet")
    raise AssertionError("should have refused")
except PermissionError as e:
    print("write guard OK:", e)


In [ ]:
# --- Semantic stubs (TTL / SHACL) ---
import s3fs

def read_text_uri(uri: str) -> str:
    if uri.startswith("s3://"):
        fs = s3fs.S3FileSystem(
            key=os.environ.get("AWS_ACCESS_KEY_ID", "admin"),
            secret=os.environ.get("AWS_SECRET_ACCESS_KEY", "admin"),
            client_kwargs={"endpoint_url": S3_ENDPOINT} if S3_ENDPOINT else {},
            config_kwargs={"s3": {"addressing_style": "path"}, "signature_version": "s3v4"},
        )
        path = uri.replace("s3://", "")
        with fs.open(path, "rb") as f:
            return f.read().decode("utf-8")
    return Path(uri.removeprefix("file://")).read_text()

if result.semantic_uris.get("catalog.ttl"):
    cat = read_text_uri(result.semantic_uris["catalog.ttl"])
    print(cat[:1200])
    assert "@context" not in cat
    assert "dcat:Dataset" in cat
    print("\\n--- shapes.ttl (head) ---")
    print(read_text_uri(result.semantic_uris["shapes.ttl"])[:800])
else:
    print("no semantic stubs (empty registration?)")


In [ ]:
# --- Pointer table preview ---
import pandas as pd
rows = meta.load_pointer_rows()
df = pd.DataFrame(rows)
print(df.head(10))
print(f"\\nrows={len(df)}  columns={list(df.columns)}")


## Next

* Layout adapters for customer prefixes (`flat_prefix`, custom).
* Full PyIceberg table commits under the warehouse.
* Java `FormatModel` (Iceberg 1.11+) mirroring `hdf5_iceberg.format`.
* Verify: feed `catalog.ttl` / shapes through a SHACL verifier.
